<a href="https://colab.research.google.com/github/mandara-samanalee/FYP_Team_Intellicore/blob/main/notebooks/module3_activity/V2_Val_%26_Test_Split_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3: Val & Test Split Feature Extraction (Rolling Window) - V2

## 📌 Overview
This notebook processes the **Validation** and **Test** splits from our combined dataset to extract rolling window burstiness features. It uses strict chronological slicing to prevent **data leakage** by preserving historic review patterns.

### 📥 Inputs
* **`df_combined_split.csv`**: Combined dataset containing real and synthetic reviews split into `train`, `val`, and `test` intervals, alongside review timestamps (`timestamp`) and unique identifier columns (`product_id`).
* **Chronological History**: Historically correct contexts mapping each product's timeline (e.g., `train` used as background for `val` calculations).

### 🔄 Context & Timeline Dependencies
* **Val Features**: Extracted using the **Train** set as historical context.
* **Test Features**: Extracted using both the **Train + Val** sets combined as historical context.

## 🛠 Feature Specifications
For each product, a **30-day window** (sliding by **7 days**) is analyzed to extract:
1. `review_count`: Volume of reviews in the window.
2. `mean_iat` & `std_iat`: Inter-Arrival Time statistics.
3. `burstiness`: Normalized index representing review spikes (-1 to 1).
4. `burst_span_5`: Shortest timespan containing 5 reviews.
5. `has_synthetic`: Target flag (1 if the window contains a known synthetic review, else 0).

## 📤 Outputs
* **`features_val_test.csv`**: A single integrated CSV file holding the computed sliding-window features for both validation and test datasets, separated by a `split` identifier column.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np

BASE  = "/content/drive/MyDrive"
FINAL = f"{BASE}/Module3_Final"

df = pd.read_csv(f"{FINAL}/df_combined_split.csv")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nBy split:")
print(df['split'].value_counts())
print("\nBy split × is_generated:")
print(df.groupby('split')['is_generated'].value_counts())

# critical checks
train_synth = ((df['split']=='train') & (df['is_generated']==1)).sum()
print(f"\n--- CHECKS ---")
print(f"Synthetic rows in TRAIN (must be 0): {train_synth}")
print(f"Total synthetic: {(df['is_generated']==1).sum():,}")
print(f"Total real:      {(df['is_generated']==0).sum():,}")

df['dt'] = pd.to_datetime(df['timestamp'], unit='s')
for sp in ['train','val','test']:
    d = df[df['split']==sp]
    print(f"  {sp}: {d['dt'].min()} → {d['dt'].max()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (471495, 13)
Columns: ['reviewer_id', 'product_id', 'rating', 'text', 'timestamp', 'datetime', 'is_generated', 'split', 'campaign_id', 'campaign_type', 'spacing_regime', 'overlap_mode', 'target_jaccard']

By split:
split
train    324446
test      74245
val       72804
Name: count, dtype: int64

By split × is_generated:
split  is_generated
test   0                69525
       1                 4720
train  0               324446
val    0                69524
       1                 3280
Name: count, dtype: int64

--- CHECKS ---
Synthetic rows in TRAIN (must be 0): 0
Total synthetic: 8,000
Total real:      463,495
  train: 2020-01-01 00:01:01 → 2021-10-31 15:33:36
  val: 2021-10-31 15:45:49 → 2022-04-20 22:40:08
  test: 2022-04-20 22:44:02 → 2022-10-30 23:51:14


In [ ]:
WINDOW_SECS = 30*24*3600
STEP_SECS   = 7*24*3600
BURST_N     = 5

def tightest_span(wt, n=BURST_N):
    if len(wt) < n: return float(WINDOW_SECS)
    return float((wt[n-1:] - wt[:-(n-1)]).min())

def extract_split(reviews, history, split_name):
    period_start = reviews['timestamp'].min()
    rows = []
    products = reviews['product_id'].unique()
    for i, pid in enumerate(products):
        if i % 2000 == 0: print(f"  {split_name}: {i:,}/{len(products):,}")
        h = history.loc[history['product_id']==pid, 'timestamp'].values
        c = reviews[reviews['product_id']==pid]
        ts_all = np.sort(np.concatenate([h, c['timestamp'].values]))
        syn = set(c.loc[c['is_generated']==1, 'timestamp'].values.tolist())
        if len(ts_all)==0: continue
        ws, t_end = ts_all[0], ts_all[-1]
        while ws <= t_end:
            we = ws + WINDOW_SECS
            wt = ts_all[(ts_all>=ws)&(ts_all<we)]
            if we >= period_start and len(wt)>0:
                cnt=len(wt)
                if cnt>=2:
                    iat=np.diff(wt); m=float(iat.mean()); s=float(iat.std())
                    b=(s-m)/(s+m) if (s+m)>0 else 0.0
                else: m=s=b=0.0
                rows.append({'product_id':pid,
                    'window_start':pd.Timestamp(ws,unit='s'),
                    'window_end':pd.Timestamp(we,unit='s'),
                    'review_count':cnt,'mean_iat':m,'std_iat':s,'burstiness':b,
                    'burst_span_5':tightest_span(wt),
                    'has_synthetic':int(bool(syn & set(wt.tolist()))),
                    'split':split_name})
            ws += STEP_SECS
    return pd.DataFrame(rows)

In [ ]:
train = df[df['split']=='train']
val   = df[df['split']=='val']
test  = df[df['split']=='test']

print("Extracting VAL (history = train)...")
feat_val = extract_split(val, train, 'val')

print("Extracting TEST (history = train+val)...")
hist_test = pd.concat([train, val], ignore_index=True)
feat_test = extract_split(test, hist_test, 'test')

# combine into one file, split column keeps them separable
features = pd.concat([feat_val, feat_test], ignore_index=True)
features.to_csv(f"{FINAL}/features_val_test.csv", index=False)

print(f"\nTotal: {len(features):,} windows")
print(f"  val:  {(features['split']=='val').sum():,} ({feat_val['has_synthetic'].sum():,} synthetic)")
print(f"  test: {(features['split']=='test').sum():,} ({feat_test['has_synthetic'].sum():,} synthetic)")

Extracting VAL (history = train)...
  val: 0/11,487
  val: 2,000/11,487
  val: 4,000/11,487
  val: 6,000/11,487
  val: 8,000/11,487
  val: 10,000/11,487
Extracting TEST (history = train+val)...
  test: 0/11,244
  test: 2,000/11,244
  test: 4,000/11,244
  test: 6,000/11,244
  test: 8,000/11,244
  test: 10,000/11,244

Total: 303,751 windows
  val:  147,033 (3,979 synthetic)
  test: 156,718 (6,013 synthetic)
